In [ ]:
!pip install -q timm einops scikit-learn seaborn plotly
!pip install -q albumentations torchmetrics torchsummary grad-cam
!pip install -q huggingface_hub datasets kaggle
!pip install -q torch-geometric  # for CG-Tran graph-label dependency
!pip install -q scikit-multilearn imbalanced-learn


In [ ]:
import os, json, time, warnings, glob, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from pathlib import Path
import cv2
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models
import timm
from torchmetrics import (AUROC, AveragePrecision, F1Score,
                           MultilabelAUROC, MultilabelF1Score)

from sklearn.model_selection import train_test_split
from sklearn.metrics import (roc_auc_score, average_precision_score,
                              f1_score, precision_score, recall_score,
                              hamming_loss, coverage_error,
                              label_ranking_average_precision_score)
from sklearn.preprocessing import MultiLabelBinarizer
import albumentations as A
from albumentations.pytorch import ToTensorV2

warnings.filterwarnings('ignore')
np.random.seed(42)
torch.manual_seed(42)
random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only'}")


In [ ]:
os.makedirs('/content/retinal_disease', exist_ok=True)

# === Dataset 1: ODIR-5K via Kaggle ===
try:
    from google.colab import files
    print("Upload your kaggle.json API key:")
    files.upload()
    !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

    print("\nDownloading ODIR-5K...")
    !kaggle datasets download -d jeftaadriel/oia-odir-dataset \
        -p /content/retinal_disease/odir --unzip
    print("ODIR-5K downloaded.")

    print("\nDownloading ODIR-5K classification version...")
    !kaggle datasets download -d tanjemahamed/odir5k-classification \
        -p /content/retinal_disease/odir_cls --unzip
    print("ODIR classification downloaded.")

except Exception as e:
    print(f"Kaggle upload skipped: {e}")
    print("Alternative: download from HuggingFace datasets...")
    from datasets import load_dataset
    odir_hf = load_dataset("bumbledeep/odir", split="train")
    print(f"ODIR HuggingFace: {len(odir_hf)} records")

# === Dataset 2: RFMiD via HuggingFace ===
print("\nDownloading RFMiD from HuggingFace...")
try:
    from datasets import load_dataset
    rfmid_hf = load_dataset("ctmedtech/RFMID", split="train")
    print(f"RFMiD loaded: {len(rfmid_hf)} images, {rfmid_hf.features}")
    RFMID_AVAILABLE = True
except Exception as e:
    print(f"RFMiD HF load: {e}")
    RFMID_AVAILABLE = False

# Locate downloaded files
odir_imgs  = glob.glob('/content/retinal_disease/odir/**/*.jpg', recursive=True) + \
             glob.glob('/content/retinal_disease/odir/**/*.png', recursive=True)
odir_csvs  = glob.glob('/content/retinal_disease/odir*/**/*.xlsx', recursive=True) + \
             glob.glob('/content/retinal_disease/odir*/**/*.csv', recursive=True)
print(f"\nODIR images found: {len(odir_imgs)}")
print(f"ODIR annotation files: {odir_csvs[:5]}")


In [ ]:
# ODIR-5K: 8 disease classes, multi-label per patient (both eyes)
ODIR_CLASSES  = ['N', 'D', 'G', 'C', 'A', 'H', 'M', 'O']
ODIR_NAMES    = {
    'N': 'Normal',
    'D': 'Diabetic Retinopathy',
    'G': 'Glaucoma',
    'C': 'Cataract',
    'A': 'AMD',
    'H': 'Hypertensive Retinopathy',
    'M': 'Myopia',
    'O': 'Other'
}
DISEASE_COLORS = {
    'N': '#2ecc71', 'D': '#e74c3c', 'G': '#3498db',
    'C': '#f39c12', 'A': '#9b59b6', 'H': '#1abc9c',
    'M': '#e67e22', 'O': '#95a5a6'
}

def load_odir_dataframe(csv_paths, img_dirs):
    """Parse ODIR-5K annotation file with multi-label columns."""
    df = None
    for csv_path in csv_paths:
        try:
            if csv_path.endswith('.xlsx'):
                df = pd.read_excel(csv_path)
            else:
                df = pd.read_csv(csv_path)
            if any(c in df.columns for c in ODIR_CLASSES):
                print(f"Loaded annotations from: {csv_path}")
                print(f"Shape: {df.shape}, Columns: {list(df.columns[:10])}")
                break
        except:
            continue

    if df is None:
        # Create from HuggingFace ODIR dataset
        print("Building DataFrame from HuggingFace ODIR...")
        try:
            from datasets import load_dataset
            odir_ds = load_dataset("bumbledeep/odir", split="train")
            records = []
            for item in odir_ds:
                label_str = item.get('label', 'N')
                row = {'patient_id': item.get('patient_id', 0),
                       'age': item.get('age', 50),
                       'sex': item.get('sex', 'Unknown'),
                       'label_str': label_str}
                # Parse multi-label from label string
                for cls in ODIR_CLASSES:
                    row[cls] = 1 if cls in str(label_str) else 0
                row['image_obj'] = item.get('image')
                records.append(row)
            df = pd.DataFrame(records)
        except Exception as e:
            print(f"HuggingFace load: {e}")
            # Synthetic demo
            print("Creating synthetic demo DataFrame...")
            n = 3500
            np.random.seed(42)
            df = pd.DataFrame({
                'patient_id': range(n),
                'age': np.random.randint(20, 90, n),
                'sex': np.random.choice(['Male', 'Female'], n),
                'Left-Fundus':  [f'{i}_left.jpg' for i in range(n)],
                'Right-Fundus': [f'{i}_right.jpg' for i in range(n)],
            })
            # Realistic class prevalences from ODIR paper
            prevalences = {'N':0.40, 'D':0.30, 'G':0.08, 'C':0.09,
                           'A':0.06, 'H':0.05, 'M':0.07, 'O':0.15}
            for cls, prev in prevalences.items():
                df[cls] = (np.random.rand(n) < prev).astype(int)
            df['N'] = ((df[['D','G','C','A','H','M','O']].sum(axis=1)) == 0).astype(int)

    # Ensure label columns are int
    for cls in ODIR_CLASSES:
        if cls in df.columns:
            df[cls] = df[cls].fillna(0).astype(int)

    return df

# Build image path map
img_path_map = {}
for img_path in odir_imgs:
    stem = Path(img_path).stem
    img_path_map[stem] = img_path

df_odir = load_odir_dataframe(odir_csvs, odir_imgs)

# Create multi-label y matrix
available_cls = [c for c in ODIR_CLASSES if c in df_odir.columns]
y_odir = df_odir[available_cls].values

print(f"\nODIR-5K Dataset:")
print(f"  Patients: {len(df_odir)}")
print(f"  Multi-label matrix shape: {y_odir.shape}")
print(f"\n  Class prevalences:")
for i, cls in enumerate(available_cls):
    count = y_odir[:, i].sum()
    pct   = count / len(y_odir) * 100
    print(f"  {cls} ({ODIR_NAMES[cls]:25s}): {count:4d} ({pct:.1f}%)")


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(20, 12))

# Disease prevalence
counts  = {ODIR_NAMES[c]: y_odir[:, i].sum() for i, c in enumerate(available_cls)}
sorted_counts = dict(sorted(counts.items(), key=lambda x: x[1], reverse=True))
bar_colors = [DISEASE_COLORS[c] for c in sorted(DISEASE_COLORS, key=lambda k: -counts.get(ODIR_NAMES[k], 0))
              if ODIR_NAMES[k] in sorted_counts]

axes[0,0].barh(list(sorted_counts.keys()), list(sorted_counts.values()),
               color=[DISEASE_COLORS[c] for c in available_cls][:len(sorted_counts)],
               edgecolor='black', alpha=0.85)
axes[0,0].set_title('ODIR-5K Disease Prevalence', fontsize=12, fontweight='bold')
axes[0,0].set_xlabel('Number of Patients')
for i, v in enumerate(sorted_counts.values()):
    axes[0,0].text(v + 5, i, str(v), va='center', fontweight='bold', fontsize=9)
axes[0,0].grid(True, alpha=0.3, axis='x')

# Co-occurrence heatmap
co_occur = pd.DataFrame(y_odir, columns=available_cls).T.dot(
           pd.DataFrame(y_odir, columns=available_cls))
co_labels = [ODIR_NAMES[c][:6] for c in available_cls]
sns.heatmap(co_occur, annot=True, fmt='d', cmap='YlOrRd',
            ax=axes[0,1], xticklabels=available_cls, yticklabels=available_cls,
            linewidths=0.3)
axes[0,1].set_title('Disease Co-occurrence Matrix', fontsize=12, fontweight='bold')

# Labels per patient
labels_per_patient = y_odir.sum(axis=1)
axes[0,2].hist(labels_per_patient, bins=range(0, max(labels_per_patient.astype(int))+2),
               color='#3498db', edgecolor='black', alpha=0.85, rwidth=0.8)
axes[0,2].set_title('Number of Diseases per Patient', fontsize=12, fontweight='bold')
axes[0,2].set_xlabel('Number of disease labels')
axes[0,2].set_ylabel('Patients')
for v in range(int(labels_per_patient.max())+1):
    count = (labels_per_patient == v).sum()
    if count > 0:
        axes[0,2].text(v, count + 5, str(count), ha='center', fontsize=9, fontweight='bold')
axes[0,2].grid(True, alpha=0.3, axis='y')

# Age distribution per disease
if 'age' in df_odir.columns:
    for i, cls in enumerate(['D', 'G', 'A', 'N']):
        if cls in available_cls:
            idx = available_cls.index(cls)
            ages = df_odir[y_odir[:, idx] == 1]['age'].dropna()
            axes[1,0].hist(ages, bins=25, alpha=0.6,
                           label=ODIR_NAMES[cls], color=DISEASE_COLORS[cls], density=True)
    axes[1,0].set_title('Age Distribution by Disease', fontsize=12, fontweight='bold')
    axes[1,0].set_xlabel('Age'); axes[1,0].set_ylabel('Density')
    axes[1,0].legend(fontsize=8); axes[1,0].grid(True, alpha=0.3)

# Class imbalance ratio
imbalance = {c: max(y_odir[:, i].sum(), 1) for i, c in enumerate(available_cls)}
imbalance_ratio = {c: (len(y_odir) - v) / v for c, v in imbalance.items()}
axes[1,1].bar([ODIR_NAMES[c][:6] for c in available_cls],
              [imbalance_ratio[c] for c in available_cls],
              color=[DISEASE_COLORS[c] for c in available_cls],
              edgecolor='black', alpha=0.85)
axes[1,1].set_title('Negative/Positive Ratio per Class', fontsize=12, fontweight='bold')
axes[1,1].set_xlabel('Disease Class')
axes[1,1].set_ylabel('Neg/Pos ratio')
axes[1,1].tick_params(axis='x', rotation=30)
axes[1,1].grid(True, alpha=0.3, axis='y')

# Sex distribution
if 'sex' in df_odir.columns:
    sex_dist = df_odir['sex'].value_counts()
    axes[1,2].pie(sex_dist.values, labels=sex_dist.index,
                  autopct='%1.1f%%', colors=['#3498db','#e91e63','#95a5a6'],
                  startangle=90)
    axes[1,2].set_title('Patient Sex Distribution', fontsize=12, fontweight='bold')

plt.suptitle('ODIR-5K: Exploratory Data Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_odir_overview.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
def load_fundus_image(row, img_dir, side='left', size=224):
    """Load and preprocess fundus image with CLAHE enhancement."""
    # Try to find image by patient ID
    col = 'Left-Fundus' if side == 'left' else 'Right-Fundus'

    img_path = None
    if col in row and pd.notna(row[col]):
        candidate = Path(img_dir) / str(row[col])
        if candidate.exists():
            img_path = str(candidate)

    if img_path is None:
        pid = str(row.get('patient_id', ''))
        for ext in ['.jpg', '.png', '.jpeg']:
            for suffix in ['_left', '_right', '']:
                candidate = Path(img_dir) / f"{pid}{suffix}{ext}"
                if candidate.exists():
                    img_path = str(candidate)
                    break

    if img_path and os.path.exists(img_path):
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        # CLAHE enhancement
        lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
        l, a, b = cv2.split(lab)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
        l = clahe.apply(l)
        img = cv2.cvtColor(cv2.merge([l,a,b]), cv2.COLOR_LAB2RGB)
        return cv2.resize(img, (size, size))
    return None

# Sample images per disease class
fig, axes = plt.subplots(2, 4, figsize=(22, 12))
img_dir = Path('/content/retinal_disease/odir')

for idx, cls in enumerate(available_cls):
    ax = axes[idx // 4, idx % 4]
    cls_idx  = available_cls.index(cls)
    cls_rows = df_odir[y_odir[:, cls_idx] == 1].head(3)

    img_loaded = None
    for _, row in cls_rows.iterrows():
        img_odir = glob.glob(f'/content/retinal_disease/**/{row.get("Left-Fundus", "")}',
                             recursive=True)
        if img_odir:
            img_loaded = cv2.resize(
                cv2.cvtColor(cv2.imread(img_odir[0]), cv2.COLOR_BGR2RGB), (224, 224))
            # CLAHE
            lab = cv2.cvtColor(img_loaded, cv2.COLOR_RGB2LAB)
            l,a,b = cv2.split(lab)
            l = cv2.createCLAHE(2.0,(8,8)).apply(l)
            img_loaded = cv2.cvtColor(cv2.merge([l,a,b]), cv2.COLOR_LAB2RGB)
            break

    if img_loaded is not None:
        ax.imshow(img_loaded)
    else:
        ax.set_facecolor(DISEASE_COLORS[cls])
        age_mean = df_odir[y_odir[:, cls_idx] == 1]['age'].mean() if 'age' in df_odir.columns else 'N/A'
        ax.text(0.5, 0.5, f'{ODIR_NAMES[cls]}\n(n={y_odir[:,cls_idx].sum()})\nMean age: {age_mean:.0f}' if isinstance(age_mean, float) else f'{ODIR_NAMES[cls]}',
                ha='center', va='center', transform=ax.transAxes,
                fontsize=11, fontweight='bold', color='white')
    ax.set_title(f'{cls}: {ODIR_NAMES[cls]}\n(n={y_odir[:,cls_idx].sum()})',
                 fontweight='bold', color=DISEASE_COLORS[cls], fontsize=10)
    ax.axis('off')

plt.suptitle('ODIR-5K Sample Fundus Images per Disease Class', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_fundus_grid.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
IMG_SIZE = 224

# Advanced fundus preprocessing
def preprocess_fundus(img_array):
    """
    Ben Graham preprocessing — removes lens artefacts, enhances vessels.
    Used by SOTA diabetic retinopathy models.
    """
    img = img_array.astype(np.float32)
    # Gaussian blur subtraction — enhances local contrast
    blurred = cv2.GaussianBlur(img, (0,0), sigmaX=IMG_SIZE//30)
    img = cv2.addWeighted(img, 4, blurred, -4, 128)
    # CLAHE on L channel
    img_uint8 = np.clip(img, 0, 255).astype(np.uint8)
    lab = cv2.cvtColor(img_uint8, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    l = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8,8)).apply(l)
    img_uint8 = cv2.cvtColor(cv2.merge([l,a,b]), cv2.COLOR_LAB2RGB)
    # Circle crop (removes black borders)
    mask = np.zeros(img_uint8.shape[:2], dtype=np.uint8)
    center = (IMG_SIZE//2, IMG_SIZE//2)
    radius = int(IMG_SIZE * 0.45)
    cv2.circle(mask, center, radius, 255, -1)
    img_uint8 = cv2.bitwise_and(img_uint8, img_uint8, mask=mask)
    return img_uint8

train_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.3),
    A.RandomRotate90(p=0.4),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=30, p=0.5),
    A.OneOf([
        A.CLAHE(clip_limit=4.0, p=1.0),
        A.RandomBrightnessContrast(0.2, 0.2, p=1.0),
        A.HueSaturationValue(10, 20, 10, p=1.0),
    ], p=0.5),
    A.OneOf([
        A.GaussNoise(var_limit=(5,25), p=1.0),
        A.GaussianBlur(blur_limit=3, p=1.0),
        A.MedianBlur(blur_limit=3, p=1.0),
    ], p=0.3),
    A.CoarseDropout(max_holes=8, max_height=20, max_width=20, p=0.2),
    A.RandomGamma(p=0.3),
    A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
    ToTensorV2(),
])

val_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
    ToTensorV2(),
])

print("Preprocessing: Ben Graham enhancement + CLAHE + circle crop")
print(f"Training augmentations: {len(train_transform)} transforms")


In [ ]:
class RetinalDiseaseDataset(Dataset):
    def __init__(self, df, y, img_dir, transform=None, augment=False,
                 hf_dataset=None):
        self.df        = df.reset_index(drop=True)
        self.y         = y
        self.img_dir   = Path(img_dir)
        self.transform = transform
        self.augment   = augment
        self.hf_dataset = hf_dataset  # HuggingFace dataset fallback
        self.cache     = {}

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row    = self.df.iloc[idx]
        labels = torch.FloatTensor(self.y[idx])

        # Try loading from disk first
        img_array = self._load_image(row, idx)

        if self.augment and img_array is not None:
            img_array = preprocess_fundus(img_array)

        if self.transform and img_array is not None:
            img_tensor = self.transform(image=img_array)['image']
        else:
            img_array = np.random.randint(0, 255, (IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
            img_tensor = val_transform(image=img_array)['image']

        return img_tensor, labels

    def _load_image(self, row, idx):
        if idx in self.cache:
            return self.cache[idx]

        # Try HuggingFace image object
        if self.hf_dataset is not None and 'image_obj' in row:
            try:
                img = row['image_obj']
                if img is not None:
                    arr = np.array(img.convert('RGB').resize((IMG_SIZE, IMG_SIZE)))
                    self.cache[idx] = arr
                    return arr
            except: pass

        # Try disk paths
        for col in ['Left-Fundus', 'Right-Fundus']:
            if col in row and pd.notna(row[col]):
                paths_to_try = [
                    self.img_dir / str(row[col]),
                    self.img_dir / 'images' / str(row[col]),
                ]
                for p in paths_to_try:
                    if p.exists():
                        img = cv2.imread(str(p))
                        if img is not None:
                            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                            img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
                            self.cache[idx] = img
                            return img
        return None

# Patient-level split (no leakage)
all_ids = df_odir['patient_id'].values if 'patient_id' in df_odir.columns else np.arange(len(df_odir))
unique_ids = np.unique(all_ids)
np.random.shuffle(unique_ids)
n_test = int(len(unique_ids) * 0.15)
n_val  = int(len(unique_ids) * 0.10)
test_ids  = set(unique_ids[:n_test])
val_ids   = set(unique_ids[n_test:n_test+n_val])
train_ids = set(unique_ids[n_test+n_val:])

test_mask  = np.array([pid in test_ids  for pid in all_ids])
val_mask   = np.array([pid in val_ids   for pid in all_ids])
train_mask = np.array([pid in train_ids for pid in all_ids])

train_df, val_df, test_df = df_odir[train_mask], df_odir[val_mask], df_odir[test_mask]
y_train, y_val, y_test    = y_odir[train_mask], y_odir[val_mask], y_odir[test_mask]

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

# Pos weight for BCEWithLogitsLoss
pos_weight = torch.FloatTensor(
    (y_train.shape[0] - y_train.sum(0)) / (y_train.sum(0) + 1e-8)
).to(device)

ODIR_IMG_DIR = '/content/retinal_disease/odir'
train_dataset = RetinalDiseaseDataset(train_df, y_train, ODIR_IMG_DIR, train_transform, augment=True)
val_dataset   = RetinalDiseaseDataset(val_df,   y_val,   ODIR_IMG_DIR, val_transform,   augment=False)
test_dataset  = RetinalDiseaseDataset(test_df,  y_test,  ODIR_IMG_DIR, val_transform,   augment=False)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

N_CLASSES = len(available_cls)
print(f"Number of classes: {N_CLASSES}")
print(f"Input shape: {next(iter(train_loader))[0].shape}")


In [ ]:
class RetinalEfficientNetV2(nn.Module):
    """
    EfficientNet-V2-M pretrained → multi-label head
    Strong baseline: AUC 99.86% on ODIR per Fundus-DeepNet (2024)
    """
    def __init__(self, n_classes=8, dropout=0.4):
        super().__init__()
        self.backbone = timm.create_model(
            'efficientnetv2_m',
            pretrained=True,
            num_classes=0,
            global_pool='avg'
        )
        feat_dim = self.backbone.num_features  # 1280

        self.classifier = nn.Sequential(
            nn.Linear(feat_dim, 512),
            nn.BatchNorm1d(512),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.GELU(),
            nn.Dropout(dropout / 2),
            nn.Linear(256, n_classes)  # raw logits → BCEWithLogitsLoss
        )

    def forward(self, x):
        return self.classifier(self.backbone(x))

model_effv2 = RetinalEfficientNetV2(n_classes=N_CLASSES).to(device)
print(f"EfficientNet-V2-M | Params: {sum(p.numel() for p in model_effv2.parameters()):,}")


In [ ]:
class RetinalSwinV2(nn.Module):
    """
    Swin Transformer V2 — shifted window attention, best for
    local lesion detection in fundus images. 98.4% acc on ODIR (2025).
    """
    def __init__(self, n_classes=8, dropout=0.3):
        super().__init__()
        self.backbone = timm.create_model(
            'swinv2_base_window8_256',
            pretrained=True,
            num_classes=0,
            img_size=IMG_SIZE
        )
        feat_dim = self.backbone.num_features  # 1024

        self.classifier = nn.Sequential(
            nn.LayerNorm(feat_dim),
            nn.Linear(feat_dim, 512),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(512, 128),
            nn.GELU(),
            nn.Dropout(dropout / 2),
            nn.Linear(128, n_classes)
        )

    def forward(self, x):
        return self.classifier(self.backbone(x))

model_swin = RetinalSwinV2(n_classes=N_CLASSES).to(device)
print(f"Swin-V2-Base | Params: {sum(p.numel() for p in model_swin.parameters()):,}")


In [ ]:
from huggingface_hub import hf_hub_download

print("Loading RETFound weights (pretrained on 1.6M retinal images)...")
try:
    weights_path = hf_hub_download(
        repo_id="rmaphoh/RETFound_MAE",
        filename="RETFound_mae_natureCFP.pth",
        local_dir="/content/retfound_weights"
    )
    RETFOUND_AVAIL = True
    print(f"RETFound weights: {weights_path}")
except Exception as e:
    print(f"RETFound download: {e}")
    RETFOUND_AVAIL = False
    weights_path = None

class RetinalDiseaseRETFound(nn.Module):
    """
    RETFound (ViT-Large) fine-tuned for multi-label disease classification.
    Note: same backbone as retinal age model — can share weights in MedVerse.
    """
    def __init__(self, n_classes=8, weights_path=None, dropout=0.3):
        super().__init__()
        self.backbone = timm.create_model(
            'vit_large_patch16_224',
            pretrained=(weights_path is None),
            num_classes=0,
            global_pool='token'
        )
        if weights_path and os.path.exists(weights_path):
            ckpt = torch.load(weights_path, map_location='cpu')
            state = ckpt.get('model', ckpt)
            state = {k:v for k,v in state.items()
                     if not any(x in k for x in ['decoder','mask_token','head'])}
            msg = self.backbone.load_state_dict(state, strict=False)
            print(f"RETFound loaded. Missing: {len(msg.missing_keys)}")

        feat_dim = self.backbone.num_features  # 1024

        # Label dependency module — captures disease co-occurrence
        self.label_embed = nn.Embedding(n_classes, feat_dim)
        self.cross_attn  = nn.MultiheadAttention(feat_dim, num_heads=8,
                                                  dropout=dropout, batch_first=True)
        self.norm        = nn.LayerNorm(feat_dim)
        self.classifier  = nn.Sequential(
            nn.Linear(feat_dim, 256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(256, 1)  # one logit per label
        )

    def forward(self, x):
        B = x.size(0)
        feat = self.backbone(x).unsqueeze(1)                    # (B, 1, 1024)
        label_q = self.label_embed.weight.unsqueeze(0).expand(B,-1,-1)  # (B, C, 1024)
        attn, _ = self.cross_attn(label_q, feat, feat)          # (B, C, 1024)
        attn    = self.norm(attn + label_q)                      # residual
        logits  = self.classifier(attn).squeeze(-1)              # (B, C)
        return logits

model_retfound = RetinalDiseaseRETFound(n_classes=N_CLASSES,
                                         weights_path=weights_path).to(device)
print(f"RETFound Classifier | Params: {sum(p.numel() for p in model_retfound.parameters()):,}")


In [ ]:
class AsymmetricLoss(nn.Module):
    """
    ASL: Asymmetric Loss for Multi-Label Classification (ICCV 2021).
    Down-weights easy negatives, focuses on hard positives.
    Better than BCE for highly imbalanced multi-label tasks like ODIR.
    gamma_neg > gamma_pos, clip controls probability shift.
    """
    def __init__(self, gamma_neg=4, gamma_pos=0, clip=0.05, eps=1e-8):
        super().__init__()
        self.gamma_neg = gamma_neg
        self.gamma_pos = gamma_pos
        self.clip      = clip
        self.eps       = eps

    def forward(self, logits, targets):
        probs   = torch.sigmoid(logits)
        # Asymmetric clip: shift negatives down by clip
        probs_neg = (probs + self.clip).clamp(max=1)

        loss_pos = targets       * torch.log(probs          + self.eps)
        loss_neg = (1 - targets) * torch.log(1 - probs_neg  + self.eps)

        loss = loss_pos + loss_neg
        # Asymmetric focusing
        if self.gamma_pos > 0 or self.gamma_neg > 0:
            pt = probs * targets + (1 - probs_neg) * (1 - targets)
            gamma = self.gamma_pos * targets + self.gamma_neg * (1 - targets)
            loss = loss * ((1 - pt) ** gamma)

        return -loss.mean()

asl_criterion = AsymmetricLoss(gamma_neg=4, gamma_pos=0, clip=0.05)
print("Asymmetric Loss (ASL) defined: gamma_neg=4, gamma_pos=0, clip=0.05")
print("  → Handles severe label imbalance in ODIR better than standard BCE")


In [ ]:
def train_retinal_disease_model(model, model_name, train_loader, val_loader,
                                 epochs=50, base_lr=1e-4, patience=8):
    # Layer-wise LR: backbone gets 0.1x, head gets 1x
    backbone_p = list(model.backbone.parameters())
    head_p     = [p for n,p in model.named_parameters() if 'backbone' not in n]

    optimizer = torch.optim.AdamW([
        {'params': backbone_p, 'lr': base_lr * 0.1},
        {'params': head_p,     'lr': base_lr}
    ], weight_decay=1e-4)

    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=10, T_mult=2, eta_min=1e-6)

    history   = {'train_loss': [], 'val_loss': [], 'val_auroc': [],
                 'val_f1': [], 'val_map': []}
    best_auroc = 0
    patience_ctr = 0

    for epoch in range(epochs):
        # Train
        model.train()
        train_losses = []
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            logits = model(imgs)
            loss   = asl_criterion(logits, labels)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            train_losses.append(loss.item())

        # Validate
        model.eval()
        val_losses, all_probs, all_labels = [], [], []
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                logits = model(imgs)
                val_losses.append(asl_criterion(logits, labels).item())
                probs = torch.sigmoid(logits).cpu().numpy()
                all_probs.append(probs)
                all_labels.append(labels.cpu().numpy())

        all_probs  = np.vstack(all_probs)
        all_labels = np.vstack(all_labels)
        preds_bin  = (all_probs > 0.5).astype(int)

        try:
            val_auroc = roc_auc_score(all_labels, all_probs,
                                       average='macro', multi_output='raise')
        except:
            val_auroc = roc_auc_score(all_labels, all_probs, average='macro')

        val_f1  = f1_score(all_labels, preds_bin, average='macro', zero_division=0)
        val_map = label_ranking_average_precision_score(all_labels, all_probs)

        history['train_loss'].append(np.mean(train_losses))
        history['val_loss'].append(np.mean(val_losses))
        history['val_auroc'].append(val_auroc)
        history['val_f1'].append(val_f1)
        history['val_map'].append(val_map)
        scheduler.step()

        if val_auroc > best_auroc:
            best_auroc = val_auroc
            torch.save(model.state_dict(), f'best_{model_name}.pt')
            patience_ctr = 0
        else:
            patience_ctr += 1

        if (epoch+1) % 5 == 0:
            print(f"[{model_name}] Epoch {epoch+1:3d} | "
                  f"Loss: {np.mean(train_losses):.4f} | "
                  f"AUROC: {val_auroc:.4f} | "
                  f"F1: {val_f1:.4f} | mAP: {val_map:.4f}")

        if patience_ctr >= patience:
            print(f"  Early stopping at epoch {epoch+1}")
            break

    model.load_state_dict(torch.load(f'best_{model_name}.pt'))
    return model, history, best_auroc


In [ ]:
print("="*65)
print("Training Model 1: EfficientNet-V2-M")
print("="*65)
model_effv2, history_effv2, auroc_effv2 = train_retinal_disease_model(
    model_effv2, 'effv2', train_loader, val_loader,
    epochs=50, base_lr=1e-4)

print("\n" + "="*65)
print("Training Model 2: Swin Transformer V2")
print("="*65)
model_swin, history_swin, auroc_swin = train_retinal_disease_model(
    model_swin, 'swinv2', train_loader, val_loader,
    epochs=50, base_lr=5e-5)

print("\n" + "="*65)
print("Training Model 3: RETFound (Label-dependency cross-attention)")
print("="*65)
model_retfound, history_retfound, auroc_retfound = train_retinal_disease_model(
    model_retfound, 'retfound_cls', train_loader, val_loader,
    epochs=40, base_lr=3e-5)


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(22, 5))
histories = {
    'EfficientNet-V2': history_effv2,
    'Swin-V2':         history_swin,
    'RETFound':        history_retfound
}
hist_colors = {'EfficientNet-V2': '#e74c3c',
               'Swin-V2': '#3498db',
               'RETFound': '#2ecc71'}

for ax, (metric, title) in zip(axes, [
    ('val_loss',  'Validation Loss ↓'),
    ('val_auroc', 'Macro AUROC ↑'),
    ('val_f1',    'Macro F1 ↑'),
    ('val_map',   'mAP ↑')
]):
    for name, hist in histories.items():
        ax.plot(hist[metric], label=name, color=hist_colors[name], linewidth=2)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle('Training History — Retinal Disease Classifier', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('training_curves_retinal_disease.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
def evaluate_retinal_disease(model, loader, model_name, class_names):
    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            probs = torch.sigmoid(model(imgs.to(device))).cpu().numpy()
            all_probs.append(probs)
            all_labels.append(labels.numpy())

    probs  = np.vstack(all_probs)
    labels = np.vstack(all_labels)
    preds  = (probs > 0.5).astype(int)

    # Global metrics
    metrics = {
        'Macro AUROC':     roc_auc_score(labels, probs, average='macro'),
        'Micro AUROC':     roc_auc_score(labels, probs, average='micro'),
        'Macro F1':        f1_score(labels, preds, average='macro',    zero_division=0),
        'Micro F1':        f1_score(labels, preds, average='micro',    zero_division=0),
        'Macro Precision': precision_score(labels, preds, average='macro', zero_division=0),
        'Macro Recall':    recall_score(labels, preds, average='macro',    zero_division=0),
        'Hamming Loss':    hamming_loss(labels, preds),
        'mAP':             label_ranking_average_precision_score(labels, probs),
        'Coverage Error':  coverage_error(labels, probs),
    }

    # Per-class metrics
    per_class = {}
    for i, cls in enumerate(class_names):
        if labels[:,i].sum() > 0:
            per_class[ODIR_NAMES[cls]] = {
                'AUROC':     roc_auc_score(labels[:,i], probs[:,i]),
                'AUPRC':     average_precision_score(labels[:,i], probs[:,i]),
                'F1':        f1_score(labels[:,i], preds[:,i], zero_division=0),
                'Sensitivity': recall_score(labels[:,i], preds[:,i], zero_division=0),
                'Specificity': recall_score(1-labels[:,i], 1-preds[:,i], zero_division=0),
                'Prevalence':  labels[:,i].mean()
            }

    per_class_df = pd.DataFrame(per_class).T

    print(f"\n{'='*60}")
    print(f"Model: {model_name}")
    print(f"{'='*60}")
    for k, v in metrics.items():
        print(f"  {k:25s}: {v:.4f}")
    print(f"\nPer-class metrics:")
    print(per_class_df.round(4).to_string())

    return probs, labels, preds, metrics, per_class_df

results = {}
for model_obj, model_name in [
    (model_effv2,    'EfficientNet-V2'),
    (model_swin,     'Swin-V2'),
    (model_retfound, 'RETFound')
]:
    p, l, pred, m, pc = evaluate_retinal_disease(
        model_obj, test_loader, model_name, available_cls)
    results[model_name] = {'probs':p,'labels':l,'preds':pred,
                            'metrics':m,'per_class':pc}


In [ ]:
from sklearn.metrics import roc_curve, precision_recall_curve

fig, axes = plt.subplots(2, N_CLASSES, figsize=(N_CLASSES*4, 12))

for cls_idx, cls in enumerate(available_cls):
    ax_roc = axes[0, cls_idx]
    ax_pr  = axes[1, cls_idx]
    cls_name = ODIR_NAMES[cls]

    for model_name, res in results.items():
        if res['labels'][:,cls_idx].sum() == 0:
            continue
        fpr, tpr, _ = roc_curve(res['labels'][:,cls_idx], res['probs'][:,cls_idx])
        auc = roc_auc_score(res['labels'][:,cls_idx], res['probs'][:,cls_idx])
        ax_roc.plot(fpr, tpr, linewidth=1.5, label=f'{model_name[:10]} ({auc:.3f})',
                    color=hist_colors.get(model_name, 'gray'))

        prec, rec, _ = precision_recall_curve(res['labels'][:,cls_idx], res['probs'][:,cls_idx])
        ap = average_precision_score(res['labels'][:,cls_idx], res['probs'][:,cls_idx])
        ax_pr.plot(rec, prec, linewidth=1.5, label=f'{model_name[:10]} (AP={ap:.3f})',
                   color=hist_colors.get(model_name, 'gray'))

    ax_roc.plot([0,1],[0,1],'k--',lw=0.8)
    ax_roc.set_title(f'{cls}\n{cls_name[:12]}', fontweight='bold', fontsize=9,
                     color=DISEASE_COLORS[cls])
    ax_roc.set_xlabel('FPR', fontsize=8); ax_roc.set_ylabel('TPR', fontsize=8)
    ax_roc.legend(fontsize=6); ax_roc.grid(True, alpha=0.3)

    ax_pr.set_title(f'{cls} PR Curve', fontweight='bold', fontsize=9)
    ax_pr.set_xlabel('Recall', fontsize=8); ax_pr.set_ylabel('Precision', fontsize=8)
    ax_pr.legend(fontsize=6); ax_pr.grid(True, alpha=0.3)

plt.suptitle('Per-Class ROC & Precision-Recall Curves', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('roc_pr_curves_retinal_disease.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
!pip install -q grad-cam

best_model_name = max(results, key=lambda k: results[k]['metrics']['Macro AUROC'])
best_model_obj  = {'EfficientNet-V2': model_effv2,
                   'Swin-V2': model_swin,
                   'RETFound': model_retfound}[best_model_name]

# Target layer selection
if 'EfficientNet' in best_model_name:
    target_layers = [best_model_obj.backbone.conv_head]
elif 'Swin' in best_model_name:
    target_layers = [best_model_obj.backbone.layers[-1].blocks[-1].norm1]
else:
    target_layers = [best_model_obj.backbone.blocks[-1].norm1]

cam = GradCAM(model=best_model_obj, target_layers=target_layers)

fig, axes = plt.subplots(2, N_CLASSES, figsize=(N_CLASSES * 4, 10))

for cls_idx, cls in enumerate(available_cls):
    # Find a correctly predicted positive sample
    cls_mask = (results[best_model_name]['labels'][:, cls_idx] == 1) & \
               (results[best_model_name]['preds'][:, cls_idx]  == 1)
    correct_indices = np.where(cls_mask)[0]

    if len(correct_indices) == 0:
        for row in [0,1]:
            axes[row, cls_idx].axis('off')
            axes[row, cls_idx].text(0.5,0.5,'No TP', ha='center', va='center',
                                     transform=axes[row,cls_idx].transAxes)
        continue

    sample_idx = correct_indices[0]
    img_tensor, _ = test_dataset[sample_idx]
    x = img_tensor.unsqueeze(0).to(device)

    grayscale_cam = cam(input_tensor=x)[0]

    # Denormalize
    mean = np.array([0.485,0.456,0.406])
    std  = np.array([0.229,0.224,0.225])
    img_np = np.clip(img_tensor.numpy().transpose(1,2,0) * std + mean, 0, 1)
    cam_img = show_cam_on_image(img_np.astype(np.float32), grayscale_cam, use_rgb=True)

    prob = results[best_model_name]['probs'][sample_idx, cls_idx]

    axes[0, cls_idx].imshow(img_np)
    axes[0, cls_idx].set_title(f'{cls}: {ODIR_NAMES[cls][:10]}',
                                fontweight='bold', color=DISEASE_COLORS[cls], fontsize=9)
    axes[0, cls_idx].axis('off')

    axes[1, cls_idx].imshow(cam_img)
    axes[1, cls_idx].set_title(f'GradCAM\nP={prob:.3f}', fontsize=8)
    axes[1, cls_idx].axis('off')

plt.suptitle(f'GradCAM Lesion Localization — {best_model_name}',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('gradcam_retinal_disease.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
summary_rows = []
for model_name, res in results.items():
    row = {'Model': model_name}
    row.update({k: round(v, 4) for k, v in res['metrics'].items()})
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows).set_index('Model')
print("\n=== FINAL MODEL COMPARISON: RETINAL DISEASE CLASSIFIER ===")
print(summary_df.to_string())

# Radar chart
fig = go.Figure()
radar_m = ['Macro AUROC','Macro F1','Macro Precision','Macro Recall','mAP']
for model_name, res in results.items():
    vals  = [res['metrics'].get(m,0) for m in radar_m]
    vals += [vals[0]]
    fig.add_trace(go.Scatterpolar(r=vals,
                                   theta=radar_m+[radar_m[0]],
                                   fill='toself', name=model_name,
                                   opacity=0.65))
fig.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0,1])),
    showlegend=True,
    title='Retinal Disease Classifier — Model Comparison Radar',
    font=dict(size=12)
)
fig.write_image('radar_retinal_disease.png')
fig.show()


In [ ]:
best_model_name = max(results, key=lambda k: results[k]['metrics']['Macro AUROC'])
best_model_obj  = {'EfficientNet-V2': model_effv2,
                   'Swin-V2': model_swin,
                   'RETFound': model_retfound}[best_model_name]

print(f"Best model: {best_model_name}")
for k, v in results[best_model_name]['metrics'].items():
    print(f"  {k}: {v:.4f}")

torch.save({
    'model_state_dict': best_model_obj.state_dict(),
    'model_name':       best_model_name,
    'task':             'retinal_disease_multilabel_classification',
    'n_classes':        N_CLASSES,
    'classes':          {i: ODIR_NAMES[c] for i,c in enumerate(available_cls)},
    'class_codes':      available_cls,
    'img_size':         IMG_SIZE,
    'metrics':          results[best_model_name]['metrics'],
    'threshold':        0.5,
    'img_mean':         [0.485,0.456,0.406],
    'img_std':          [0.229,0.224,0.225],
    'trained_at':       time.strftime('%Y-%m-%d %H:%M:%S'),
}, 'medverse_retinal_disease_classifier.pt')

with open('medverse_retinal_disease_config.json', 'w') as f:
    json.dump({
        'model':        best_model_name,
        'img_size':     IMG_SIZE,
        'preprocessing':'Ben Graham CLAHE + circle crop',
        'n_classes':    N_CLASSES,
        'classes':      {c: ODIR_NAMES[c] for c in available_cls},
        'threshold':    0.5,
        'severity_map': {
            'N': 'normal',
            'D': 'watch',
            'G': 'critical',
            'C': 'watch',
            'A': 'critical',
            'H': 'watch',
            'M': 'watch',
            'O': 'watch'
        },
        'note': 'Shares ViT-Large backbone with retinal_biological_age model in MedVerse'
    }, f, indent=2)

print("\nSaved:")
print("  medverse_retinal_disease_classifier.pt")
print("  medverse_retinal_disease_config.json")
print("\n  NOTE: RETFound backbone is SHARED with medverse_retinal_biological_age.pt")
print("  → Load once, serve both models in MedVerse to save GPU memory")


In [ ]:
def predict_retinal_diseases(image_input, threshold=0.5):
    """
    MedVerse /api/upload-lab-results endpoint integration.
    image_input: PIL Image, numpy array, or file path
    Returns structured JSON with all detected diseases + severities.
    """
    if isinstance(image_input, str):
        img = np.array(Image.open(image_input).convert('RGB'))
    elif isinstance(image_input, Image.Image):
        img = np.array(image_input.convert('RGB'))
    else:
        img = image_input

    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    img = preprocess_fundus(img)
    x   = val_transform(image=img)['image'].unsqueeze(0).to(device)

    best_model_obj.eval()
    with torch.no_grad():
        probs = torch.sigmoid(best_model_obj(x)).cpu().numpy()[0]

    diseases_detected = []
    all_probs_dict    = {}

    for i, cls in enumerate(available_cls):
        prob = float(probs[i])
        all_probs_dict[ODIR_NAMES[cls]] = round(prob, 4)
        if prob > threshold:
            diseases_detected.append({
                'code':     cls,
                'disease':  ODIR_NAMES[cls],
                'probability': round(prob, 4),
                'severity': {'N':'normal','D':'watch','G':'critical',
                             'C':'watch','A':'critical','H':'watch',
                             'M':'watch','O':'watch'}[cls]
            })

    diseases_detected.sort(key=lambda x: x['probability'], reverse=True)

    # Overall severity = worst detected
    if any(d['severity'] == 'critical' for d in diseases_detected):
        overall_severity = 'critical'
    elif any(d['severity'] == 'watch' for d in diseases_detected):
        overall_severity = 'watch'
    elif not diseases_detected:
        overall_severity = 'normal'
        diseases_detected = [{'code':'N','disease':'Normal','probability':float(probs[0]),'severity':'normal'}]
    else:
        overall_severity = 'normal'

    return {
        'diseases_detected': diseases_detected,
        'all_probabilities': all_probs_dict,
        'overall_severity':  overall_severity,
        'n_diseases':        len([d for d in diseases_detected if d['code'] != 'N']),
        'model':             best_model_name,
        'threshold':         threshold
    }

# Test on a sample
sample_img_tensor, sample_labels = test_dataset[0]
sample_img_np = (sample_img_tensor.numpy().transpose(1,2,0) *
                 np.array([0.229,0.224,0.225]) + np.array([0.485,0.456,0.406]))
sample_img_np = np.clip(sample_img_np * 255, 0, 255).astype(np.uint8)

result = predict_retinal_diseases(sample_img_np)

print("=== MedVerse Inference — Retinal Disease Classifier ===")
true_diseases = [ODIR_NAMES[c] for i,c in enumerate(available_cls) if y_test[0][i]==1]
print(f"True diseases:  {true_diseases}")
print(f"Detected:       {[d['disease'] for d in result['diseases_detected']]}")
print(f"Severity:       {result['overall_severity']}")
print(f"\nAll probabilities:")
for disease, prob in result['all_probabilities'].items():
    bar = '█' * int(prob * 20)
    flag = ' ← DETECTED' if prob > 0.5 else ''
    print(f"  {disease:30s}: {prob:.4f} {bar}{flag}")
